# RiskModels API — Quickstart

Get hedge ratios, factor risk, and Deep Dive snapshots for any US equity.
Paste your API key below and hit **Runtime → Run all**.
Get a key at [riskmodels.app/get-key](https://riskmodels.app/get-key).

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import os, requests, pandas as pd
from IPython.display import display, Image

# Colab secrets use google.colab.userdata, not os.environ
def _get_secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name)

API_KEY = (
    _get_secret("RISKMODELS_API_KEY")
    or _get_secret("RISKMODELS_QUICKSTART_API_KEY")
    or _get_secret("TEST_API_KEY")
    or "PASTE_YOUR_KEY_HERE"  # <-- paste your key here if not using secrets
)
BASE_URL = (
    (_get_secret("RISKMODELS_QUICKSTART_BASE_URL") or "https://riskmodels.app")
    .rstrip("/") + "/api"
)

session = requests.Session()
session.headers["Authorization"] = f"Bearer {API_KEY}"

r = session.get(f"{BASE_URL}/balance", params={"include_usage": "false"})
r.raise_for_status()
print(f"Connected. Key: {API_KEY[:16]}...")

---
## Precision Hedge Chart

See how a stock's return decomposes into market, sector, and subsector factor legs over time.

In [ ]:
# ── Precision Hedge Chart ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

TICKER = "NVDA"   # Try it: change to "AAPL", "TSLA", or "JPM"
YEARS  = 3        # Try it: set to 1 or 5 to zoom in/out

resp = session.get(
    f"{BASE_URL}/ticker-returns",
    params={"ticker": TICKER, "years": YEARS},
)
resp.raise_for_status()
data = resp.json().get("data", [])
df = pd.DataFrame(data)

if df.empty:
    print(f"No data for {TICKER}.")
else:
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    # Geometric Bridge (see docs/methodology — "Multi-Period Return Attribution").
    # prod_L*(t) = Π(1 + combined factor return through level *) at date t.
    prod_l1 = (1.0 + df["l1_cfr"].fillna(0.0)).cumprod()
    prod_l2 = (1.0 + df["l2_cfr"].fillna(0.0)).cumprod()
    prod_l3 = (1.0 + df["l3_cfr"].fillna(0.0)).cumprod()
    prod_g  = (1.0 + df["returns_gross"].fillna(0.0)).cumprod()

    # Line chart: cumulative return "if only factors through this level existed".
    df["cum_stock"]      = prod_g  - 1.0
    df["cum_through_l1"] = prod_l1 - 1.0
    df["cum_through_l2"] = prod_l2 - 1.0
    df["cum_through_l3"] = prod_l3 - 1.0

    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor("#0d0d0d")
    ax.set_facecolor("#0d0d0d")

    colors = {
        "cum_stock":      "#60a5fa",
        "cum_through_l1": "#6366f1",
        "cum_through_l2": "#34d399",
        "cum_through_l3": "#9ca3af",
    }
    labels = {
        "cum_stock":      TICKER,
        "cum_through_l1": "Through L1 (market)",
        "cum_through_l2": "Through L2 (+ sector)",
        "cum_through_l3": "Through L3 (+ subsector)",
    }

    for col, color in colors.items():
        ax.plot(df["date"], df[col], color=color, linewidth=1.4, label=labels[col])

    ax.axhline(0, color="#444", linewidth=0.8, linestyle="--")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
    ax.tick_params(colors="#aaa", labelsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333")
    ax.set_xlabel("Date", color="#aaa", fontsize=9)
    ax.set_ylabel("Cumulative Return", color="#aaa", fontsize=9)
    ax.set_title(
        f"Your Precision Hedge Recipe — {TICKER}  ({YEARS}y)",
        color="white", fontsize=12, pad=10
    )
    ax.legend(
        frameon=False, labelcolor="#ccc", fontsize=9,
        loc="upper left", title="Series", title_fontsize=8,
    )
    ax.grid(axis="y", color="#222", linewidth=0.6)
    plt.tight_layout()
    plt.show()

    # Geometric Bridge waterfall bars — telescoping differences that sum to gross.
    pl1, pl2, pl3, pg = prod_l1.iloc[-1], prod_l2.iloc[-1], prod_l3.iloc[-1], prod_g.iloc[-1]
    bars = {
        "Market (L1)":            pl1 - 1.0,
        "Sector (L2 − L1)":       pl2 - pl1,
        "Subsector (L3 − L2)":    pl3 - pl2,
        "Residual (G − L3)":      pg  - pl3,
        f"{TICKER} gross (= sum)": pg  - 1.0,
    }
    summary = pd.DataFrame({"Value": {k: f"{v * 100:.1f}%" for k, v in bars.items()}})
    print(f"Geometric Bridge attribution over {YEARS}y — as of {df['date'].iloc[-1].date()}")
    display(summary)

---
## Risk Snapshot — Hedge Ratios & Volatility

One API call gives you hedge ratios at three precision levels, plus volatility and explained risk.

In [ ]:
# ── Risk Snapshot ────────────────────────────────────────────────────────────────────
ticker = "NVDA"  # Try it: change to "AAPL", "TSLA", "JPM"

resp = session.get(f"{BASE_URL}/metrics/{ticker}")
resp.raise_for_status()
m = resp.json()
metrics = m.get("metrics", m)
meta = m.get("meta", {})

# Helpers so the table gracefully handles None / sub-1% values.
def _hr(key):
    v = metrics.get(key)
    return round(v, 4) if v is not None else None

def _er(key):
    v = metrics.get(key)
    # 2 decimals so micro-caps of ER (e.g. NVDA's subsector ER ≈ 0.07%) stay
    # visible — the chart below rounds to 0% otherwise and looks like a bug.
    return f"{v:.2%}" if v is not None else None

snapshot = pd.DataFrame({"Value": {
    "Date":            m.get("teo"),
    "Price":           metrics.get("price_close"),
    "Vol (23d)":       round(metrics["vol_23d"], 4) if metrics.get("vol_23d") else None,
    "Sector ETF":      meta.get("sector_etf"),
    "Subsector ETF":   meta.get("subsector_etf"),
    # Hedge ratios — regression coefficients; sign/magnitude show direction & sensitivity.
    "L1 Market HR":    _hr("l1_mkt_hr"),
    "L2 Sector HR":    _hr("l2_sec_hr"),
    "L3 Subsector HR": _hr("l3_sub_hr"),
    # Explained Risk — variance shares at L3 (sum to ≈ 100%); plotted in the chart below.
    "L3 Market ER":    _er("l3_mkt_er"),
    "L3 Sector ER":    _er("l3_sec_er"),
    "L3 Subsector ER": _er("l3_sub_er"),
    "L3 Residual ER":  _er("l3_res_er"),
}})
print(f"{ticker} \u2014 Risk Snapshot")
display(snapshot)

# ── Visual: L3 Explained Risk decomposition ──
# HR ≠ ER: HR is a regression coefficient (can be negative), ER is the variance
# share (0–1, sums to ~1). The chart below shows ER. A small subsector ER (e.g.
# NVDA's ~0.07%) is genuine, not a bug — it means very little of NVDA's risk is
# explained by SMH *after* market and sector exposure are already hedged out.
er_items = [
    ("Market",    metrics.get("l3_mkt_er"), "#60a5fa"),
    ("Sector",    metrics.get("l3_sec_er"), "#6366f1"),
    ("Subsector", metrics.get("l3_sub_er"), "#a78bfa"),
    ("Residual",  metrics.get("l3_res_er"), "#f87171"),
]
er_items = [(l, v, c) for l, v, c in er_items if v is not None]

def _fmt_pct(v):
    """Smart % formatter — show 2 decimals for values under 1%, so a genuine
    0.07% reads as '0.07%' instead of '0%' (which looks like missing data)."""
    av = abs(v)
    if av < 0.01:
        return f"{v:.2%}"
    if av < 0.1:
        return f"{v:.1%}"
    return f"{v:.0%}"

if er_items:
    fig, ax = plt.subplots(figsize=(8, 2.5))
    fig.patch.set_facecolor("#0d0d0d")
    ax.set_facecolor("#0d0d0d")
    labs, vals, cols = zip(*er_items)
    ax.barh(labs, vals, color=cols, edgecolor="#444", height=0.5)
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
    ax.tick_params(colors="#aaa", labelsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333")
    max_val = max(vals)
    for i, v in enumerate(vals):
        # Place label to the right of the bar; pad 1% of the max so it clears
        # the bar edge even for tiny bars.
        ax.text(v + max_val * 0.01, i, _fmt_pct(v), va="center", color="#ccc", fontsize=9)
    ax.set_title(f"{ticker} \u2014 L3 Explained Risk Decomposition", color="white", fontsize=10, pad=8)
    plt.tight_layout()
    plt.show()

---
## Deep Dive Snapshot

A precomputed risk report as a PNG image — one API call, no rendering code.

In [ ]:
# ── Deep Dive Snapshot ──────────────────────────────────────────────────────────────
ticker = "NVDA"  # Try it: change to "AAPL", "MSFT", "GOOGL"

resp = session.get(f"{BASE_URL}/snapshot/{ticker}", params={"format": "png"})
if resp.status_code == 404:
    print(f"No snapshot for {ticker} (not yet in coverage).")
else:
    resp.raise_for_status()
    display(Image(data=resp.content))

---
## Portfolio Hedge

Submit weighted positions and get back hedge ratios at every level in one batch call.

In [ ]:
# ── Portfolio Hedge ──────────────────────────────────────────────────────────
# Try it: change tickers and weights to your own portfolio (weights should sum to 1.0)
portfolio = {
    "AAPL":  0.25,
    "MSFT":  0.20,
    "NVDA":  0.20,
    "GOOGL": 0.15,
    "AMZN":  0.10,
    "JPM":   0.10,
}

resp = session.post(
    f"{BASE_URL}/batch/analyze",
    json={"tickers": list(portfolio.keys()), "metrics": ["hedge_ratios"], "years": 1},
)
resp.raise_for_status()
results = resp.json()["results"]

rows = []
for ticker, weight in portfolio.items():
    r  = results.get(ticker, {})
    hr = r.get("hedge_ratios") or {}
    rows.append({
        "ticker":       ticker,
        "weight":       weight,
        "l1_market_hr": hr.get("l1_market"),
        "l2_market_hr": hr.get("l2_market"),
        "l2_sector_hr": hr.get("l2_sector"),
        "l3_market_hr": hr.get("l3_market"),
        "l3_sector_hr": hr.get("l3_sector"),
        "l3_sub_hr":    hr.get("l3_subsector"),
    })

df_port = pd.DataFrame(rows).set_index("ticker")

hr_cols = ["l1_market_hr", "l2_market_hr", "l2_sector_hr",
           "l3_market_hr", "l3_sector_hr", "l3_sub_hr"]
any_null = df_port[hr_cols].isnull().all()
if any_null.any():
    missing = [c for c in hr_cols if any_null[c]]
    print(f"Note: API returned null for {missing} — batch/analyze may need full_metrics instead.")
    print("Falling back to per-ticker /metrics calls...\n")
    for ticker in portfolio:
        mr = session.get(f"{BASE_URL}/metrics/{ticker}")
        if mr.status_code == 200:
            mj = mr.json()
            met = mj.get("metrics", mj)
            df_port.loc[ticker, "l1_market_hr"] = met.get("l1_mkt_hr")
            df_port.loc[ticker, "l2_market_hr"] = met.get("l2_mkt_hr")
            df_port.loc[ticker, "l2_sector_hr"] = met.get("l2_sec_hr")
            df_port.loc[ticker, "l3_market_hr"] = met.get("l3_mkt_hr")
            df_port.loc[ticker, "l3_sector_hr"] = met.get("l3_sec_hr")
            df_port.loc[ticker, "l3_sub_hr"]    = met.get("l3_sub_hr")

def _wtd(col):
    s = df_port[col].dropna()
    if s.empty:
        return None
    return round((df_port["weight"] * df_port[col].fillna(0)).sum(), 4)

port_summary = pd.DataFrame({
    "Market HR": [_wtd("l1_market_hr"), _wtd("l2_market_hr"), _wtd("l3_market_hr")],
    "Sector HR": ["—", _wtd("l2_sector_hr"), _wtd("l3_sector_hr")],
    "Subsector HR": ["—", "—", _wtd("l3_sub_hr")],
}, index=["L1", "L2", "L3"]).rename_axis("Level")

print("Portfolio-level hedge ratios (weighted):")
display(port_summary)

print("\nPer-position breakdown:")
display(df_port[["weight", "l1_market_hr", "l2_market_hr",
                  "l2_sector_hr", "l3_market_hr", "l3_sector_hr", "l3_sub_hr"]])

---
## Portfolio Diversification

How much risk does your portfolio actually retain after accounting for cross-sector correlations?

In [ ]:
# ── Portfolio Diversification ──────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

resp = session.post(
    f"{BASE_URL}/portfolio/risk-snapshot",
    json={
        "positions": [{"ticker": t, "weight": w} for t, w in portfolio.items()],
        "format": "json",
        "include_diversification": True,
        "window_days": 252,
        "no_cache": True,
    },
)
resp.raise_for_status()
snap = resp.json()

div = snap.get("portfolio_risk_index", {}).get("diversification")
if not div:
    print("Diversification not available (API may still be deploying).")
else:
    layers = div["layers"]
    labels = [l["layer"].title() for l in layers]
    naive  = [l["naive_er"] for l in layers]
    adj    = [l["adjusted_er"] for l in layers]
    mults  = [l.get("multiplier") or 0 for l in layers]

    # Consistent layer colors (bottom to top in stacked: market, sector, subsector, residual)
    LAYER_COLORS = {"Market": "#60a5fa", "Sector": "#6366f1", "Subsector": "#a78bfa", "Residual": "#f87171"}

    fig, axes = plt.subplots(1, 3, figsize=(15, 5.5),
                             gridspec_kw={"width_ratios": [2, 2, 1.2]})
    fig.patch.set_facecolor("#0d0d0d")
    fig.suptitle("Portfolio Diversification Analysis", color="white", fontsize=13, y=0.98, fontweight="bold")
    for ax in axes:
        ax.set_facecolor("#0d0d0d")
        for spine in ax.spines.values():
            spine.set_edgecolor("#333")
        ax.tick_params(colors="#aaa", labelsize=8)

    # ── Panel 1: side-by-side bars ──
    x = np.arange(len(labels))
    bw = 0.35
    ax1 = axes[0]
    ax1.bar(x - bw/2, naive, bw, color="#6366f1", edgecolor="#444", alpha=0.85, label="Naive")
    ax1.bar(x + bw/2, adj, bw, color="#34d399", edgecolor="#444", alpha=0.85, label="Adjusted")
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, fontsize=8)
    ax1.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
    ax1.set_title("Naive vs Adjusted ER", color="white", fontsize=10, pad=8)
    ax1.set_ylabel("Explained Risk", color="#aaa", fontsize=8)
    ax1.grid(axis="y", color="#222", linewidth=0.5)
    ax1.legend(frameon=False, labelcolor="#ccc", fontsize=8,
              loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)

    # ── Panel 2: stacked before/after ──
    ax2 = axes[1]
    bar_labels = ["Naive\n(weighted sum)", "Adjusted\n(corr-adjusted)"]
    x2 = np.arange(2)
    bottoms = np.zeros(2)
    for lbl, n_val, a_val in zip(labels, naive, adj):
        vals = [n_val, a_val]
        c = LAYER_COLORS[lbl]
        ax2.bar(x2, vals, 0.5, bottom=bottoms, color=c, edgecolor="#0d0d0d", linewidth=0.5,
                label=lbl)
        for j in range(2):
            if vals[j] > 0.025:
                ax2.text(x2[j], bottoms[j] + vals[j]/2, f"{vals[j]:.0%}",
                         ha="center", va="center", color="white", fontsize=8, fontweight="bold")
        bottoms += vals
    ax2.set_xticks(x2)
    ax2.set_xticklabels(bar_labels, fontsize=8)
    ax2.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
    ax2.set_title("Stacked Risk Decomposition", color="white", fontsize=10, pad=8)
    ax2.legend(frameon=False, labelcolor="#ccc", fontsize=7,
              loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)

    # ── Panel 3: retained fraction ──
    ax3 = axes[2]
    ret_colors = [LAYER_COLORS[l] for l in labels]
    ax3.barh(labels, mults, color=ret_colors, edgecolor="#444", height=0.5, alpha=0.85)
    ax3.set_xlim(0, 1.15)
    ax3.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
    ax3.axvline(1.0, color="#555", linewidth=0.8, linestyle="--")
    ax3.set_title("Risk Retained", color="white", fontsize=10, pad=8)
    ax3.invert_yaxis()
    for i, m in enumerate(mults):
        ax3.text(m + 0.02, i, f"{m:.0%}", va="center", color="#ccc", fontsize=8)

    plt.tight_layout(rect=[0, 0.06, 1, 0.94])
    plt.show()

    # ── Summary table ──
    cr = div["diversification_credit"]
    summary = pd.DataFrame({"Value": {
        "Naive total ER":       f"{div['naive_pws']['total']:.1%}",
        "Adjusted total ER":    f"{div['correlation_adjusted']['total']:.1%}",
        "Diversification credit": f"{cr['total']:.1%}",
        "Sector credit":        f"{cr['sector']:.1%}",
        "Residual credit":      f"{cr['residual']:.1%}",
        "Window":               f"{div['window_days']}d",
    }})
    display(summary)

---
## Sharpe Ratios — Gross vs Hedged Residuals

Strip out market, sector, and subsector factor returns to isolate the **idiosyncratic alpha**.
The residual return at each hedge level often has a **higher Sharpe ratio** than the gross —
especially during periods when the broad market is flat.

In [ ]:
# ── Sharpe Ratios: Gross vs Hedged Residuals ─────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

TICKER = "NVDA"   # Try it: change to any ticker
YEARS  = 3        # Try it: 1, 3, or 5

resp = session.get(
    f"{BASE_URL}/ticker-returns",
    params={"ticker": TICKER, "years": YEARS},
)
resp.raise_for_status()
data = resp.json().get("data", [])
df = pd.DataFrame(data)

if df.empty:
    print(f"No data for {TICKER}.")
else:
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)
    for c in ["returns_gross", "l1_cfr", "l2_cfr", "l3_cfr"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Daily arithmetic residuals — used for Sharpe stats (standard daily Sharpe input).
    if "l1_cfr" in df.columns:
        df["l1_rr"] = df["returns_gross"] - df["l1_cfr"]
    if "l2_cfr" in df.columns:
        df["l2_rr"] = df["returns_gross"] - df["l2_cfr"]
    if "l3_cfr" in df.columns:
        df["l3_rr"] = df["returns_gross"] - df["l3_cfr"]

    # Cumulative residual SERIES follow the Geometric Bridge: prod_G(t) − prod_L*(t).
    # Do NOT compound the daily residual series directly — that drifts from the waterfall bar.
    prod_g = (1.0 + df["returns_gross"].fillna(0.0)).cumprod()
    cum_map = {"returns_gross": prod_g - 1.0}
    for lvl, col in [("l1", "l1_cfr"), ("l2", "l2_cfr"), ("l3", "l3_cfr")]:
        if col in df.columns:
            prod_l = (1.0 + df[col].fillna(0.0)).cumprod()
            cum_map[f"{lvl}_rr"] = prod_g - prod_l

    ANNUALIZE = np.sqrt(252)
    series_map = {
        "Gross":                       "returns_gross",
        "L1 residual (− market)": "l1_rr",
        "L2 residual (− sector)": "l2_rr",
        "L3 residual (− subsec)": "l3_rr",
    }
    series_colors = {
        "Gross": "#60a5fa",
        "L1 residual (− market)": "#6366f1",
        "L2 residual (− sector)": "#a78bfa",
        "L3 residual (− subsec)": "#34d399",
    }

    def sharpe(s):
        s = s.dropna()
        if len(s) < 20 or s.std() == 0:
            return np.nan
        return (s.mean() / s.std()) * ANNUALIZE

    # ── Quarterly Sharpe heatmap (daily residuals) ──
    df["quarter"] = df["date"].dt.to_period("Q")
    quarters = sorted(df["quarter"].unique())
    q_labels = [str(q) for q in quarters]
    s_labels = [l for l in series_map if series_map[l] in df.columns]

    sharpe_grid = np.full((len(s_labels), len(quarters)), np.nan)
    for qi, q in enumerate(quarters):
        mask = df["quarter"] == q
        for si, label in enumerate(s_labels):
            col = series_map[label]
            if col in df.columns:
                sharpe_grid[si, qi] = sharpe(df.loc[mask, col].astype(float))

    fig, (ax_heat, ax_cum) = plt.subplots(2, 1, figsize=(13, 9),
                                           gridspec_kw={"height_ratios": [1, 1.3]})
    fig.patch.set_facecolor("#0d0d0d")
    fig.suptitle(f"{TICKER} — Sharpe Ratios by Quarter ({YEARS}y)",
                 color="white", fontsize=13, y=0.98, fontweight="bold")

    # Heatmap
    ax_heat.set_facecolor("#0d0d0d")
    finite = sharpe_grid[np.isfinite(sharpe_grid)]
    vmax = min(4, max(1, float(np.nanmax(np.abs(finite))) if finite.size else 1.0))
    im = ax_heat.imshow(sharpe_grid, aspect="auto", cmap="RdYlGn",
                        vmin=-vmax, vmax=vmax, interpolation="nearest")
    ax_heat.set_yticks(range(len(s_labels)))
    ax_heat.set_yticklabels(s_labels, color="#ccc", fontsize=9)
    ax_heat.set_xticks(range(len(q_labels)))
    ax_heat.set_xticklabels(q_labels, color="#aaa", fontsize=7, rotation=45, ha="right")
    for si in range(len(s_labels)):
        for qi in range(len(q_labels)):
            v = sharpe_grid[si, qi]
            if np.isfinite(v):
                ax_heat.text(qi, si, f"{v:.1f}", ha="center", va="center",
                             color="black" if abs(v) < vmax*0.6 else "white", fontsize=7)
    cbar = fig.colorbar(im, ax=ax_heat, pad=0.02, shrink=0.8)
    cbar.ax.tick_params(colors="#aaa", labelsize=7)
    cbar.set_label("Annualized Sharpe", color="#aaa", fontsize=8)
    ax_heat.set_title("Quarterly Sharpe: Gross vs Each Hedge Level",
                      color="white", fontsize=10, pad=8)

    # ── Cumulative return chart (Geometric Bridge residuals) ──
    ax_cum.set_facecolor("#0d0d0d")
    for label in s_labels:
        col = series_map[label]
        if col not in cum_map:
            continue
        cr = cum_map[col]
        sr = sharpe(df[col].astype(float))
        ax_cum.plot(df["date"], cr, color=series_colors[label], linewidth=1.3,
                    label=f"{label}  (SR {sr:.2f})")

    ax_cum.axhline(0, color="#444", linewidth=0.8, linestyle="--")
    ax_cum.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
    ax_cum.tick_params(colors="#aaa", labelsize=9)
    for spine in ax_cum.spines.values():
        spine.set_edgecolor("#333")
    ax_cum.set_xlabel("Date", color="#aaa", fontsize=9)
    ax_cum.set_ylabel("Cumulative Return", color="#aaa", fontsize=9)
    ax_cum.set_title("Cumulative: Gross vs Hedged Residuals (Geometric Bridge)",
                     color="white", fontsize=10, pad=8)
    ax_cum.legend(frameon=False, labelcolor="#ccc", fontsize=8, loc="upper left")
    ax_cum.grid(axis="y", color="#222", linewidth=0.6)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

    # ── Full-period summary (daily residuals) ──
    rows = []
    for label in s_labels:
        col = series_map[label]
        s = df[col].astype(float)
        rows.append({
            "Series": label,
            "Ann. Return": f"{s.mean() * 252:.1%}",
            "Ann. Vol": f"{s.std() * ANNUALIZE:.1%}",
            "Sharpe": f"{sharpe(s):.2f}",
        })
    print(f"\nFull-period Sharpe ({YEARS}y):")
    display(pd.DataFrame(rows).set_index("Series"))

---
## Rankings — Where Does Your Stock Rank?

Cross-sectional percentile rankings across the full universe, sector, or subsector.

In [ ]:
# ── Rankings ─────────────────────────────────────────────────────────────────
ticker = "NVDA"  # Try it: change to any ticker

resp = session.get(f"{BASE_URL}/rankings/{ticker}")
resp.raise_for_status()
data = resp.json()

if data.get("rankings"):
    df_rank = pd.DataFrame(data["rankings"])
    df_rank = df_rank.dropna(subset=["rank_percentile"])
    if not df_rank.empty:
        print(f"{data['ticker']} \u2014 Rankings (as of {data.get('date', 'latest')})\n")
        display(df_rank[["display_label", "rank_percentile", "rank_ordinal", "cohort_size"]].reset_index(drop=True))
    else:
        print(f"No populated rankings for {ticker}")
else:
    print(f"No rankings for {ticker}")

---
## Factor Risk Decomposition

Break down a stock's risk into market, sector, subsector, and residual components month by month.

In [ ]:
# ── Factor Risk Table ────────────────────────────────────────────────────────────────
ticker = "NVDA"  # Try it: change to any ticker

resp = session.get(
    f"{BASE_URL}/l3-decomposition",
    params={"ticker": ticker, "market_factor_etf": "SPY"},
)
resp.raise_for_status()
body = resp.json()

df_risk = pd.DataFrame({
    "date":         pd.to_datetime(body.get("dates", [])),
    "market_er":    body.get("l3_market_er", []),
    "sector_er":    body.get("l3_sector_er", []),
    "subsector_er": body.get("l3_subsector_er", []),
    "residual_er":  body.get("l3_residual_er", []),
})
df_risk[["market_er", "sector_er", "subsector_er", "residual_er"]] = (
    df_risk[["market_er", "sector_er", "subsector_er", "residual_er"]].fillna(0)
)
df_risk = df_risk.sort_values("date").reset_index(drop=True)

if df_risk.empty:
    print(f"No factor data for {ticker}.")
else:
    df_risk["total_er"] = df_risk[["market_er", "sector_er",
                                    "subsector_er", "residual_er"]].sum(axis=1)

    df_risk["month"] = df_risk["date"].dt.to_period("M")
    df_month = df_risk.groupby("month", as_index=False).last().drop(columns=["month"])

    pct_cols = ["market_er", "sector_er", "subsector_er", "residual_er", "total_er"]
    df_month[pct_cols] = (df_month[pct_cols] * 100).round(2)
    df_month.rename(columns={
        "market_er": "market_%",
        "sector_er": "sector_%",
        "subsector_er": "subsector_%",
        "residual_er": "residual_%",
        "total_er": "total_%",
    }, inplace=True)

    print(f"Factor risk attribution for {ticker} — most recent 12 months (month-end, L3 ER → %)")
    print(f"Market ETF: {body.get('market_factor_etf', '—')}  |  Universe: {body.get('universe', '—')}")
    print()
    display(df_month.tail(12))

---
## Next Steps

- **API docs:** [riskmodels.app/docs/api](https://riskmodels.app/docs/api)
- **Python SDK:** `pip install riskmodels-py` — wraps every endpoint above into one-liners
- **Questions?** [service@riskmodels.app](mailto:service@riskmodels.app)